# Primeros modelos de mi proyecto: ventas por producto y tienda

Objetivo: predecir `UNITS`, las unidades vendidas de un producto en una tienda y un día. La preparación y la división vienen de `pipeline_proyecto.py`, así que todos los modelos ven lo mismo. La división es por fecha: entrenamiento antes del 30 de abril de 2015 y prueba desde esa fecha, porque el modelo tendría que predecir ventas futuras.

La métrica es la raíz del error cuadrático medio (RMSE), en unidades. Más abajo explico por qué no uso el error absoluto medio.

In [2]:
import time

import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.pipeline import Pipeline

from pipeline_proyecto import cargar_y_dividir, construir_preparacion

X_ent, X_prueba, y_ent, y_prueba = cargar_y_dividir()
print("Entrenamiento:", X_ent.shape, "Prueba:", X_prueba.shape)
print("Promedio de UNITS en entrenamiento:", round(y_ent.mean(), 3), "| en prueba:", round(y_prueba.mean(), 3))
print("Porcentaje de filas con 0 unidades:", round((y_ent == 0).mean() * 100, 1), "%")

Entrenamiento: (159871, 11) Prueba: (39962, 11)
Promedio de UNITS en entrenamiento: 1.059 | en prueba: 1.315
Porcentaje de filas con 0 unidades: 70.2 %


## Por qué RMSE y no error absoluto medio

Casi el 70% de las filas vende 0 unidades. Con el error absoluto medio, predecir siempre 0 es muy difícil de superar: un modelo que intenta acertar las ventas altas se equivoca un poco en las miles de filas con cero y sale peor. El RMSE castiga mucho más fallar en las ventas grandes, que son las que importan para decidir cuánto reponer. Abajo dejo las dos para la línea base, para que se vea la diferencia.

In [3]:
resultados = {}
entrenados = {}


def evaluar(nombre, modelo):
    """Entrena despues de la preparacion y mide RMSE en entrenamiento y en
    prueba. Si se vuelve a correr la celda de un modelo, reemplaza su fila."""
    pipeline = Pipeline([("preparar", construir_preparacion()), ("modelo", modelo)])
    inicio = time.perf_counter()
    pipeline.fit(X_ent, y_ent)
    segundos = time.perf_counter() - inicio
    rmse_ent = root_mean_squared_error(y_ent, pipeline.predict(X_ent))
    rmse_prueba = root_mean_squared_error(y_prueba, pipeline.predict(X_prueba))
    resultados[nombre] = {
        "modelo": nombre,
        "rmse_entrenamiento": round(rmse_ent, 4),
        "rmse_prueba": round(rmse_prueba, 4),
        "brecha": round(rmse_prueba - rmse_ent, 4),
        "segundos": round(segundos, 3),
    }
    entrenados[nombre] = pipeline
    print(f"{nombre:30s} entrenamiento {rmse_ent:7.4f}   prueba {rmse_prueba:7.4f}   {segundos:6.2f} s")

## Línea base: siempre la mediana

In [4]:
# la mediana de UNITS en entrenamiento es 0, asi que predice 0 para todo
evaluar("linea base (mediana)", DummyRegressor(strategy="median"))

pred_base = entrenados["linea base (mediana)"].predict(X_prueba)
print("MAE de la linea base en prueba:", round(mean_absolute_error(y_prueba, pred_base), 4))

linea base (mediana)           entrenamiento  3.8749   prueba  3.8348     0.19 s
MAE de la linea base en prueba: 1.3153


## Modelo lineal: sin regularizar y con Ridge

In [5]:
evaluar("lineal sin regularizar", LinearRegression())

lineal sin regularizar         entrenamiento  3.0879   prueba  3.0188     0.61 s


In [6]:
evaluar("ridge alpha=10", Ridge(alpha=10.0))

ridge alpha=10                 entrenamiento  3.0879   prueba  3.0188     0.25 s


## Ensamblajes: bosque sin límite, bosque regularizado y boosting

In [7]:
# sin ningun limite: cada arbol crece hasta dejar hojas casi puras
evaluar("bosque sin limite", RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))

bosque sin limite              entrenamiento  2.1645   prueba  3.0159    46.50 s


In [8]:
# mismo bosque, pero cada hoja tiene que juntar al menos 20 filas
evaluar("bosque min_samples_leaf=20", RandomForestRegressor(
    n_estimators=100, min_samples_leaf=20, random_state=42, n_jobs=-1))

bosque min_samples_leaf=20     entrenamiento  2.8233   prueba  2.9332    34.45 s


In [9]:
evaluar("gradient boosting", GradientBoostingRegressor(random_state=42))

gradient boosting              entrenamiento  2.8946   prueba  2.9594    20.75 s


## Tabla de resultados

In [10]:
tabla = pd.DataFrame(resultados.values())
tabla.to_csv("resultados_proyecto.csv", index=False)
tabla

,modelo,rmse_entrenamiento,rmse_prueba,brecha,segundos
0,linea base (mediana),3.8749,3.8348,-0.0401,0.192
1,lineal sin regularizar,3.0879,3.0188,-0.0691,0.611
2,ridge alpha=10,3.0879,3.0188,-0.0691,0.255
3,bosque sin limite,2.1645,3.0159,0.8514,46.498
4,bosque min_samples_leaf=20,2.8233,2.9332,0.1099,34.446
5,gradient boosting,2.8946,2.9594,0.0648,20.749


In [11]:
base = tabla.loc[tabla["modelo"] == "linea base (mediana)", "rmse_prueba"].iloc[0]
mejor = tabla.loc[tabla["rmse_prueba"].idxmin()]
print("Mejor modelo en prueba:", mejor["modelo"], "con RMSE", mejor["rmse_prueba"])
print("Mejora sobre la linea base:", round(base - mejor["rmse_prueba"], 4), "unidades,",
      round((base - mejor["rmse_prueba"]) / base * 100, 1), "%")

Mejor modelo en prueba: bosque min_samples_leaf=20 con RMSE 2.9332
Mejora sobre la linea base: 0.9016 unidades, 23.5 %


## Respuestas

Las respuestas, con los números de esta tabla, están en el `README.md` de la sesión.